# 02 · Portfolio Optimization

Notebook version of `examples/scripts/portfolio/portfolio_optimization_example.py`. Maximize portfolio yield while capping CCC exposure.



### Learning Objectives
- Build minimal USD market data for optimization
- Construct a rating-tagged bond portfolio with the fluent builder
- Optimize weights to maximize YTM subject to a CCC cap
- Inspect optimal weights and constraint values



In [ ]:
from datetime import date

from finstack.core.config import FinstackConfig
from finstack.core.currency import Currency
from finstack.core.market_data.context import MarketContext
from finstack.core.market_data.term_structures import DiscountCurve
from finstack.core.money import Money
from finstack.portfolio import (
    Entity,
    PortfolioBuilder,
    Position,
    PositionUnit,
    optimize_max_yield_with_ccc_limit,
)
from finstack.valuations.instruments import Bond


as_of = date(2025, 1, 1)


def build_market_data(as_of: date) -> MarketContext:
    """Create a simple USD discount curve for optimization examples."""
    market = MarketContext()

    usd_curve = DiscountCurve(
        "USD",
        as_of,
        [
            (0.0, 1.0),
            (1.0, 0.99),
            (3.0, 0.96),
            (5.0, 0.93),
        ],
    )
    market.insert_discount(usd_curve)

    return market


def build_bond_portfolio(as_of: date):
    """Build a USD bond portfolio with rating tags (AAA/BBB/CCC)."""
    maturity = date(as_of.year + 5, 1, 1)

    bond_aaa = (
        Bond.builder("BOND_AAA")
        .money(Money(1_000_000, "USD"))
        .issue(as_of)
        .maturity(maturity)
        .disc_id("USD")
        .coupon_rate(0.03)
        .quoted_clean_price(100.0)
        .build()
    )
    bond_bbb = (
        Bond.builder("BOND_BBB")
        .money(Money(1_000_000, "USD"))
        .issue(as_of)
        .maturity(maturity)
        .disc_id("USD")
        .coupon_rate(0.05)
        .quoted_clean_price(100.0)
        .build()
    )
    bond_ccc = (
        Bond.builder("BOND_CCC")
        .money(Money(1_000_000, "USD"))
        .issue(as_of)
        .maturity(maturity)
        .disc_id("USD")
        .coupon_rate(0.08)
        .quoted_clean_price(100.0)
        .build()
    )

    pos_aaa = (
        Position(
            "POS_AAA",
            "FUND_A",
            "BOND_AAA",
            bond_aaa,
            1.0,
            PositionUnit.FACE_VALUE,
        )
        .with_tag("rating", "AAA")
        .with_tag("sector", "investment_grade")
    )

    pos_bbb = (
        Position(
            "POS_BBB",
            "FUND_A",
            "BOND_BBB",
            bond_bbb,
            1.0,
            PositionUnit.FACE_VALUE,
        )
        .with_tag("rating", "BBB")
        .with_tag("sector", "investment_grade")
    )

    pos_ccc = (
        Position(
            "POS_CCC",
            "FUND_A",
            "BOND_CCC",
            bond_ccc,
            1.0,
            PositionUnit.FACE_VALUE,
        )
        .with_tag("rating", "CCC")
        .with_tag("sector", "high_yield")
    )

    entity = Entity("FUND_A").with_name("Example Fund")

    portfolio = (
        PortfolioBuilder("BOND_FUND_OPT")
        .name("Credit Portfolio – Optimization Example")
        .base_ccy(Currency("USD"))
        .as_of(as_of)
        .entity(entity)
        .position([pos_aaa, pos_bbb, pos_ccc])
        .build()
    )

    portfolio.validate()
    return portfolio



## 1. Build market data and the portfolio
Set up the USD curve, create rating-tagged bonds, and assemble the portfolio.



In [ ]:
market = build_market_data(as_of)
portfolio = build_bond_portfolio(as_of)

print(portfolio)
print(f"As-of: {portfolio.as_of} | Base: {portfolio.base_ccy}")
print(f"Positions: {len(portfolio.positions)}")



## 2. Optimize: maximize YTM with a CCC cap
Run the packaged helper `optimize_max_yield_with_ccc_limit` with a 20% CCC exposure limit.



In [ ]:
config = FinstackConfig()

result = optimize_max_yield_with_ccc_limit(
    portfolio,
    market,
    ccc_limit=0.20,
    strict_risk=False,
    config=config,
)

status = result["status"]
objective = result["objective_value"]
ccc_weight = result["ccc_weight"]
optimal_weights = result["optimal_weights"]

print("\n" + "=" * 80)
print("Portfolio Optimization: Maximize YTM with CCC Limit")
print("=" * 80)
print(f"Status       : {status}")
print(f"Objective YTM: {objective:.6f}")
print(f"CCC weight   : {ccc_weight:.4f} ({ccc_weight * 100:.2f}%)")

print("\nOptimal weights by position:")
for pos_id, weight in optimal_weights.items():
    print(f"  {pos_id}: {weight:.4f}")

print(f"\nWeight sum: {sum(optimal_weights.values()):.4f}")

